# Jev Box: moth orienter
Jev finds two boxes per photo: the whole moth (green) and its head (pink). The yellow line from moth centre to head gives the angle; the photo is rotated so the head points up, then square-cropped around the moth.

Edit `prompt.json` to change the target. Logic is in `boxes.py`. Put `TYPESAFE_API_KEY=...` in `moth_crop/.env`.

In [ ]:
import importlib
from pathlib import Path
from IPython.display import display, Markdown
import pandas as pd

import boxes
importlib.reload(boxes)  # pick up prompt.json / boxes.py edits without restarting

HERE = Path(boxes.__file__).parent
OUT = HERE / "output"; OUT.mkdir(exist_ok=True)
photos = sorted(p for p in (HERE / "photos").iterdir() if p.suffix.lower() in boxes.PHOTO_EXTS)
print(len(photos), "photos, target:", boxes.PROMPT["target"])

In [ ]:
rows = []
for path in photos:
    try:
        r = boxes.process(path)
    except Exception as e:
        display(Markdown(f"**{path.name}**: {e}")); continue
    r["crop"].save(OUT / f"{path.stem}_crop.jpg", quality=92)
    display(Markdown(f"### {path.name}\npresent `{r['present']}` · moth conf `{r['object_conf']}` · head conf `{r['head_conf']}` · rotated `{r['angle']}°`"))
    p = r["preview"].copy(); p.thumbnail((450, 450))
    c = r["crop"].copy(); c.thumbnail((450, 450))
    display(p, c)
    rows.append({k: v for k, v in r.items() if k not in ("preview", "crop", "raw")})
pd.DataFrame(rows)